# Rolling-Window Backtest: TC-VAE + Vine Copula Scenario Generation

Trains the TC-VAE on a fixed-length rolling window that advances 4 weeks per step.  
Each trained model generates a scenario tree CSV consumed by the C++ CVaR optimizer.

| File | Training window | Portfolio weeks |
|---|---|---|
| `tcvae_week_000.csv` | original data (ends Dec-2024) | bt weeks 0–3 |
| `tcvae_week_004.csv` | +4 bt weeks, −4 oldest weeks   | bt weeks 4–7 |
| … | … | … |
| `tcvae_week_048.csv` | +48 bt weeks, −48 oldest weeks | bt weeks 48–end |

**Rolling window**: every step drops 4 weeks from the beginning of the training  
series and appends 4 new backtest weeks, keeping the window size constant.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Paths
PROJECT_ROOT  = os.path.abspath(os.path.join(os.getcwd(), ".."))
OPTIMIZER_DIR = os.environ.get("OPTIMIZER_DIR", os.path.expanduser("~/repos/Stochastic-Portfolio-Optimizer"))
DATA_RAW      = os.path.join(PROJECT_ROOT, "data", "raw")
OPT_RAW       = os.path.join(OPTIMIZER_DIR, "data", "raw")
CKPT_BASE     = os.path.join(PROJECT_ROOT, "results", "backtest_rolling")
CSV_OUT_DIR   = os.path.join(OPTIMIZER_DIR, "data", "scenarios", "backtest")

# Data files
STOCK_NPZ  = "DOW.npz"
VIX_NPZ    = "VIX.npz"
BT_SUFFIX  = "_bt.npy"      # optimizer saves backtest as {TICKER}_bt.npy

# Rolling window
TRAIN_END_BASE = 522         # P0 row for step 0 = combined_prices[522] = Jan 2, 2025
BT_STEP        = 1          # weeks advanced per retraining step
N_STEPS        = 52          # 0,4,8,...,48  →  13 steps

# VAE architecture  (must match the checkpoint used for initialisation)
VAE_CFG = dict(
    model="BetaCVAE", encoder="CLSTMRes", decoder="CLSTMRes",
    prior="RealNVP",  conditioner="Id",
    data_dim=30,      data_length=26,
    latent_dim=8,     latent_length=26,  condition_dim=1,
    beta=0.1,         transform="log",   inv_transform="exp",
    reconstruction_loss="l1",
    C_input_dim=1,  C_hidden_dim=0,  C_num_layers=0,  C_output_dim=1,
    E_input_dim=30, E_hidden_dim=128, E_num_layers=2, E_output_dim=6,
    D_input_dim=6,  D_hidden_dim=256, D_num_layers=2, D_output_dim=30,
    P_latent_dim=6, P_num_flows=4,   P_hidden_dim=64,
    n_timestep=26,
)

# Training
EPOCHS          = 500        # epochs per retraining step (use 500 for full rigour)
WARM_START      = False       # True → load previous step's checkpoint before training
WARM_EPOCHS     = 100        # epochs when warm-starting (overrides EPOCHS for steps > 0)
LR              = 1e-3
WEIGHT_DECAY    = 1e-4
BATCH_SIZE      = 32
DEVICE          = "cuda" if __import__('torch').cuda.is_available() else "cpu"

# Scenario generation
J               = 400        # recourse nodes
E               = 40         # evaluate nodes
VINE_POOL       = 8000       # TC-VAE pool for vine copula
VINE_NEIGHBOURS = 300        # k-NN for stage-2 conditioning
VINE_TRUNC      = 5          # vine truncation level
USE_REDUCTION   = False       # True → K-medoids on J*OVERSAMPLING*E pool (Voronoi probs)
OVERSAMPLING    = 10          # pool = J * OVERSAMPLING * E

TICKERS = sorted([
    "MMM","AMZN","AXP","AMGN","AAPL","BA","CAT","CVX","CSCO","KO",
    "DIS","GS","HD","HON","IBM","JNJ","JPM","MCD","MRK","MSFT",
    "NKE","NVDA","PG","CRM","SHW","TRV","UNH","VZ","V","WMT",
])

print(f"Device : {DEVICE}")
print(f"Steps  : {N_STEPS}  ({BT_STEP} weeks each = {N_STEPS*BT_STEP} bt weeks total)")
print(f"CSVs   → {CSV_OUT_DIR}")
print(f"Checkpoints → {CKPT_BASE}")

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys, warnings
warnings.filterwarnings("ignore")

sys.path.insert(0, PROJECT_ROOT)  # repo root; importing portfolio_scenarios puts src/ on path for tsvae

import numpy as np
import pandas as pd
import torch
import torch._dynamo          # pre-initialize before pyvinecopulib can cause partial init
import torch.nn as nn
from torch.utils.data import DataLoader
from types import SimpleNamespace

from portfolio_scenarios.data_pipeline.dataset    import TCVAEDataset, DatasetOutput
from portfolio_scenarios.data_pipeline.fetcher    import load_raw_data
from portfolio_scenarios.data_pipeline.preprocess import prep_condition_vector
from tsvae.models.network_pipeline import NetworkPipeline
from portfolio_scenarios.scenario_generation.generators    import DirectTCVAE
from portfolio_scenarios.scenario_generation.tcvae_vine_csv_exporter import TCVAEVineScenarioTree

os.makedirs(CKPT_BASE,   exist_ok=True)
os.makedirs(CSV_OUT_DIR, exist_ok=True)
print("Imports OK")

In [ ]:
# ── Load original training data from DOW.npz / VIX.npz ───────────────────────
orig_tickers, orig_prices, orig_vix = load_raw_data(DATA_RAW, STOCK_NPZ, VIX_NPZ)
orig_tickers = list(orig_tickers)

# orig_prices : (522, 30)  weekly closing prices, tickers in orig_tickers order
# orig_vix    : (522, 1)   aligned VIX
orig_vix_1d = orig_vix.ravel()   # (522,)

# Map from sorted TICKERS order to orig_tickers column order
ticker_col = {t: orig_tickers.index(t) for t in TICKERS}
col_order  = [ticker_col[t] for t in TICKERS]   # re-index to sorted order
orig_prices_sorted = orig_prices[:, col_order]   # (522, 30), sorted-ticker columns

T_orig = orig_prices.shape[0]   # 522
print(f"Original data : {T_orig} weeks × {orig_prices.shape[1]} assets")
print(f"  price range : {orig_prices_sorted[0,0]:.2f} … {orig_prices_sorted[-1,0]:.2f} (AAPL first/last)")
print(f"  training rows used : 0 … {TRAIN_END_BASE-1}  (P0 = row {TRAIN_END_BASE})")

In [ ]:
# ── Load backtest prices from optimizer/data/raw/{TICKER}_bt.npy ──────────────
# Run optimizer/data/fetch_backtest_data.py first if files are missing.

bt_arrays = {}
missing_bt = []
for t in TICKERS:
    p = os.path.join(OPT_RAW, f"{t}{BT_SUFFIX}")
    if os.path.exists(p):
        bt_arrays[t] = np.load(p).astype(np.float32)
    else:
        missing_bt.append(t)

if missing_bt:
    print(f"Missing backtest files for: {missing_bt}")
    print("Downloading via yfinance...")
    import yfinance as yf
    df_bt = yf.download(missing_bt, start="2025-01-01", end="2025-12-31",
                        interval="1wk")["Close"].dropna()
    for t in missing_bt:
        if t in df_bt.columns:
            arr = df_bt[t].to_numpy().astype(np.float32)
            np.save(os.path.join(OPT_RAW, f"{t}{BT_SUFFIX}"), arr)
            bt_arrays[t] = arr
        else:
            raise RuntimeError(f"Could not fetch backtest data for {t}")

# Stack into (N_bt, 30) matrix in sorted-ticker order
N_bt_per_ticker = [len(bt_arrays[t]) for t in TICKERS]
N_bt = min(N_bt_per_ticker)   # use the shortest (all should be equal)
bt_prices = np.column_stack([bt_arrays[t][:N_bt] for t in TICKERS])  # (N_bt, 30)

print(f"Backtest data : {N_bt} weeks × {bt_prices.shape[1]} assets")
print(f"  AAPL backtest: {bt_prices[0, TICKERS.index('AAPL')]:.2f} … "
      f"{bt_prices[-1, TICKERS.index('AAPL')]:.2f}")

# Download backtest VIX
try:
    import yfinance as yf
    vix_bt_df = yf.download("^VIX", start="2025-01-01", end="2025-12-31",
                            interval="1wk")["Close"].dropna().squeeze()
    bt_vix_1d = vix_bt_df.to_numpy().astype(np.float32)[:N_bt]
    print(f"Backtest VIX  : {len(bt_vix_1d)} weeks  mean={bt_vix_1d.mean():.1f}")
except Exception as e:
    print(f"VIX download failed ({e}), using constant VIX = {orig_vix_1d[-1]:.1f}")
    bt_vix_1d = np.full(N_bt, orig_vix_1d[-1], dtype=np.float32)

In [ ]:
# ── Build combined price + VIX arrays and verify alignment ────────────────────
#
# combined_prices[t] = closing price of week t
#   rows  0 … 521  : original DOW.npz (ends Dec 26, 2024)
#   rows 522 … 521+N_bt : backtest weeks (2025)
#
# For rolling step k  (k = 0, 4, 8, … 48):
#   training prices = combined_prices[k : TRAIN_END_BASE + k]    shape (521, 30)
#   P0              = combined_prices[TRAIN_END_BASE + k]         shape (30,)
#   bt weeks used   = TRAIN_END_BASE+k … TRAIN_END_BASE+k+BT_STEP-1  (next 4 weeks)

combined_prices = np.vstack([orig_prices_sorted, bt_prices])   # (522+N_bt, 30)
combined_vix    = np.concatenate([orig_vix_1d,   bt_vix_1d])   # (522+N_bt,)

T_total = len(combined_prices)
print(f"Combined data : {T_total} total weeks")
print(f"Rolling window: {TRAIN_END_BASE} rows  ({TRAIN_END_BASE} training weeks per step)")
print()
print(f"{'Step':>4}  {'k':>3}  {'Train rows':>14}  {'P0 row':>7}  {'CSV file'}")
print("-" * 65)
for step in range(N_STEPS):
    k       = step * BT_STEP
    t_start = k
    t_end   = TRAIN_END_BASE + k    # exclusive: training = [t_start, t_end)
    p0_row  = t_end
    csv_name = f"tcvae_week_{k:03d}.csv"
    print(f"{step:>4}  {k:>3}  [{t_start:>4} … {t_end-1:>4}]    row {p0_row:>4}    {csv_name}")

max_p0_row = TRAIN_END_BASE + (N_STEPS-1)*BT_STEP
assert max_p0_row < T_total, (
    f"Not enough backtest data: need row {max_p0_row}, have {T_total}. "
    f"Fetch more backtest data or reduce N_STEPS."
)
print(f"\nData check OK — last required row: {max_p0_row} / {T_total-1} available")

In [ ]:
# ── Training helper ───────────────────────────────────────────────────────────

def build_model(cfg_dict: dict, device: str) -> nn.Module:
    cfg = SimpleNamespace(**cfg_dict)
    model = NetworkPipeline()(cfg)
    return model.to(device)


def train_model(
    model: nn.Module,
    dataset: TCVAEDataset,
    epochs: int,
    device: str,
    lr: float = LR,
    weight_decay: float = WEIGHT_DECAY,
    batch_size: int = BATCH_SIZE,
) -> nn.Module:
    """Train `model` in-place and return it."""
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    optimiser = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimiser, T_max=int(epochs * 1.5), eta_min=1e-5
    )
    model.train()

    for epoch in range(1, epochs + 1):
        epoch_loss = 0.0
        for batch in loader:
            x = batch[0].to(device)   # (B, W, Q)
            c = batch[1].to(device)   # (B, 1)
            inputs = DatasetOutput(data=x, labels=c)
            optimiser.zero_grad()
            out  = model(inputs)
            loss = out.loss
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimiser.step()
            epoch_loss += loss.item()
        scheduler.step()

        if epoch % 50 == 0 or epoch == epochs:
            avg = epoch_loss / max(len(loader), 1)
            print(f"    epoch {epoch:>4}/{epochs}  loss={avg:.5f}  "
                  f"lr={scheduler.get_last_lr()[0]:.2e}")

    model.eval()
    return model

In [ ]:
# ── Scenario generation helper ────────────────────────────────────────────────

def generate_csv(
    model: nn.Module,
    hist_windows: np.ndarray,
    hist_conds:   np.ndarray,
    hist_ret1:    np.ndarray,
    p0:           np.ndarray,
    output_path:  str,
    seed:         int = 42,
) -> None:
    """Build DirectTCVAE → TCVAEVineScenarioTree and export the scenario tree CSV."""
    tcvae_gen = DirectTCVAE(
        tcvae_model=model,
        historical_data=hist_windows,
        historical_conditions=hist_conds,
    )

    vine_tree = TCVAEVineScenarioTree(
        tcvae_generator=tcvae_gen,
        historical_returns=hist_ret1,
        initial_prices=p0,
        seed=seed,
    )

    vine_tree.fit(pool_size=VINE_POOL, trunc_lvl=VINE_TRUNC)

    vine_tree.build_tree(
        num_recourse=J,
        num_evaluate=E,
        n_sim_stage1=max(J * OVERSAMPLING, 2000),
        n_neighbours=VINE_NEIGHBOURS,
    )

    vine_tree.export_tree_to_csv(output_path)
    print(f"  CSV  → {output_path}")

In [ ]:
# ── Main rolling-window loop ──────────────────────────────────────────────────
prev_ckpt_path = None   # updated each step for warm-starting

for step in range(N_STEPS):
    k        = step * BT_STEP
    t_start  = k
    t_end    = TRAIN_END_BASE + k          # training rows [t_start, t_end)
    csv_name = f"tcvae_week_{k:03d}.csv"
    csv_path = os.path.join(CSV_OUT_DIR, csv_name)
    ckpt_dir = os.path.join(CKPT_BASE, f"step_{k:03d}")
    ckpt_path = os.path.join(ckpt_dir, "model.pt")
for run_seed in range(10):
    print(f"\n{'#'*68}")
    print(f"# GENERATING SEED FOLDER: {run_seed}")
    print(f"{'#'*68}\n")

    # P0 = prices at the START of this step's decision period.
    # Step 0: combined_prices[522] = Jan 2, 2025
    # Step 1: combined_prices[526] = Jan 30, 2025
    # Step k: combined_prices[522+k*4]
    p0 = combined_prices[t_end].astype(np.float64)
    current_csv_dir = os.path.join(CSV_OUT_DIR, str(run_seed))
    os.makedirs(current_csv_dir, exist_ok=True)

    print(f"\n{'='*68}")
    print(f" Step {step:>2} / {N_STEPS-1}  (k={k:>2})   train [{t_start}…{t_end-1}]  "
          f"P0=row {t_end}  AAPL={p0[TICKERS.index('AAPL')]:.2f}")
    print(f"{'='*68}")
    prev_ckpt_path = None   # updated each step for warm-starting

    # ── Skip if both checkpoint and CSV already exist ──────────────────────
    if os.path.exists(ckpt_path) and os.path.exists(csv_path):
        print(f"  [SKIP] checkpoint and CSV already exist.")
        prev_ckpt_path = ckpt_path
        continue
    for step in range(N_STEPS):
        k        = step * BT_STEP
        t_start  = k
        t_end    = TRAIN_END_BASE + k          # training rows [t_start, t_end)
        csv_name = f"tcvae_week_{k:03d}.csv"
        csv_path = os.path.join(current_csv_dir, csv_name)
        ckpt_dir = os.path.join(CKPT_BASE, f"step_{k:03d}")
        ckpt_path = os.path.join(ckpt_dir, "model.pt")

    # ── 1. Slice rolling window ────────────────────────────────────────────
    train_prices = combined_prices[t_start : t_end].astype(np.float32)  # (522, 30)
    train_vix    = combined_vix[t_start : t_end].astype(np.float32)     # (522,)
        # P0 = prices at the START of this step's decision period.
        # Step 0: combined_prices[522] = Jan 2, 2025
        # Step 1: combined_prices[526] = Jan 30, 2025
        # Step k: combined_prices[522+k*4]
        p0 = combined_prices[t_end].astype(np.float64)

    T_win = train_prices.shape[0]
    conditions = prep_condition_vector(train_vix, T_win)   # (T_win, 1)
        print(f"\n{'='*68}")
        print(f" Step {step:>2} / {N_STEPS-1}  (k={k:>2})   train [{t_start}…{t_end-1}]  "
              f"P0=row {t_end}  AAPL={p0[TICKERS.index('AAPL')]:.2f}")
        print(f"{'='*68}")

    dataset = TCVAEDataset(
        prices=train_prices,
        conditions=conditions,
        window_size=VAE_CFG["data_length"],
    )
    print(f"  Dataset : {len(dataset)} windows of length {VAE_CFG['data_length']}")
        # ── Skip if both checkpoint and CSV already exist ──────────────────────
        if os.path.exists(ckpt_path) and os.path.exists(csv_path):
            print(f"  [SKIP] checkpoint and CSV already exist.")
            prev_ckpt_path = ckpt_path
            continue

    # ── 2. Build model ─────────────────────────────────────────────────────
    cfg = {**VAE_CFG, "train_end_idx": T_win}
    model = build_model(cfg, DEVICE)
        # ── 1. Slice rolling window ────────────────────────────────────────────
        train_prices = combined_prices[t_start : t_end].astype(np.float32)  # (522, 30)
        train_vix    = combined_vix[t_start : t_end].astype(np.float32)     # (522,)

    # ── 3. Train ───────────────────────────────────────────────────────────
    if not os.path.exists(ckpt_path):
        warm = WARM_START and prev_ckpt_path is not None
        if warm:
            state = torch.load(prev_ckpt_path, map_location=DEVICE, weights_only=False)
        T_win = train_prices.shape[0]
        conditions = prep_condition_vector(train_vix, T_win)   # (T_win, 1)

        dataset = TCVAEDataset(
            prices=train_prices,
            conditions=conditions,
            window_size=VAE_CFG["data_length"],
        )
        print(f"  Dataset : {len(dataset)} windows of length {VAE_CFG['data_length']}")

        # ── 2. Build model ─────────────────────────────────────────────────────
        cfg = {**VAE_CFG, "train_end_idx": T_win}
        model = build_model(cfg, DEVICE)

        # ── 3. Train ───────────────────────────────────────────────────────────
        if not os.path.exists(ckpt_path):
            warm = WARM_START and prev_ckpt_path is not None
            if warm:
                state = torch.load(prev_ckpt_path, map_location=DEVICE, weights_only=False)
                if "model_state_dict" in state:
                    state = state["model_state_dict"]
                model.load_state_dict(state)
                n_epochs = WARM_EPOCHS
                print(f"  Warm-start from {prev_ckpt_path}  →  {n_epochs} epochs")
            else:
                n_epochs = EPOCHS
                print(f"  Cold-start training  →  {n_epochs} epochs")

            model = train_model(model, dataset, epochs=n_epochs, device=DEVICE)

            os.makedirs(ckpt_dir, exist_ok=True)
            torch.save({"model_state_dict": model.state_dict(), "step": step, "k": k},
                       ckpt_path)
            print(f"  Checkpoint → {ckpt_path}")
        else:
            print(f"  Loading existing checkpoint from {ckpt_path}")
            state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
            if "model_state_dict" in state:
                state = state["model_state_dict"]
            model.load_state_dict(state)
            n_epochs = WARM_EPOCHS
            print(f"  Warm-start from {prev_ckpt_path}  →  {n_epochs} epochs")
        else:
            n_epochs = EPOCHS
            print(f"  Cold-start training  →  {n_epochs} epochs")
            model.eval()

        model = train_model(model, dataset, epochs=n_epochs, device=DEVICE)
        prev_ckpt_path = ckpt_path

        os.makedirs(ckpt_dir, exist_ok=True)
        torch.save({"model_state_dict": model.state_dict(), "step": step, "k": k},
                   ckpt_path)
        print(f"  Checkpoint → {ckpt_path}")
    else:
        print(f"  Loading existing checkpoint from {ckpt_path}")
        state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        if "model_state_dict" in state:
            state = state["model_state_dict"]
        model.load_state_dict(state)
        model.eval()
        # ── 4. Generate scenario CSV ───────────────────────────────────────────
        if not os.path.exists(csv_path):
            hist_windows = dataset.data.numpy()      # (N_win, W, 30) normalised
            hist_conds   = dataset.labels.numpy()    # (N_win, 1)

    prev_ckpt_path = ckpt_path
            # All T_win-1 consecutive 1-week returns from the full training window,
            # not just the first-week return of each overlapping 26-week window.
            hist_ret1 = (train_prices[1:] / train_prices[:-1] - 1.0).astype(np.float32)

    # ── 4. Generate scenario CSV ───────────────────────────────────────────
    if not os.path.exists(csv_path):
        hist_windows = dataset.data.numpy()      # (N_win, W, 30) normalised
        hist_conds   = dataset.labels.numpy()    # (N_win, 1)
            generate_csv(
                model=model,
                hist_windows=hist_windows,
                hist_conds=hist_conds,
                hist_ret1=hist_ret1,
                p0=p0,
                output_path=csv_path,
                seed=42 + step + run_seed * 1000,
            )
        else:
            print(f"  [SKIP] CSV already exists.")

        # All T_win-1 consecutive 1-week returns from the full training window,
        # not just the first-week return of each overlapping 26-week window.
        hist_ret1 = (train_prices[1:] / train_prices[:-1] - 1.0).astype(np.float32)

        generate_csv(
            model=model,
            hist_windows=hist_windows,
            hist_conds=hist_conds,
            hist_ret1=hist_ret1,
            p0=p0,
            output_path=csv_path,
            seed=42 + step,
        )
    else:
        print(f"  [SKIP] CSV already exists.")

print(f"\n{'='*68}")
print(" Rolling backtest complete.")
print(" Rolling backtest complete for all seeds.")
print(f"{'='*68}")

In [ ]:
# ── Verification: check all 13 CSVs exist and have correct dimensions ─────────
import sys as _sys
_sys.path.insert(0, OPTIMIZER_DIR)   # must come before importing main
import importlib as _il
_main = _il.import_module("main")

Q = len(TICKERS)
expected_rows = J * E
expected_cols = 3 + 2 * Q

# Optimizer's initial_prices — must match the P0_FIXED used in the main loop
opt_prices    = {t: np.load(os.path.join(OPT_RAW, f"{t}.npy")) for t in TICKERS}
initial_prices = np.array([opt_prices[t][-1] for t in TICKERS])   # Dec 26, 2024
print(f"Optimizer P0: AAPL={initial_prices[0]:.2f}  MSFT={initial_prices[20]:.2f}  NVDA={initial_prices[22]:.2f}")
print(f"\nExpected: {N_STEPS} files, {expected_rows} rows × {expected_cols} cols each\n")
print(f"{'File':<28} {'Rows':>6} {'Cols':>5} {'p_j sum':>9} {'max_mu':>8}  Status")
print("-" * 72)

all_ok = True
for step in range(N_STEPS):
    k    = step * BT_STEP
    name = f"tcvae_week_{k:03d}.csv"
    path = os.path.join(CSV_OUT_DIR, name)
for run_seed in range(10):
    print(f"\n{'='*72}")
    print(f"Verifying Seed Folder: {run_seed}")
    print(f"{'='*72}")
    print(f"{'File':<28} {'Rows':>6} {'Cols':>5} {'p_j sum':>9} {'max_mu':>8}  Status")
    print("-" * 72)
    current_csv_dir = os.path.join(CSV_OUT_DIR, str(run_seed))
    for step in range(N_STEPS):
        k    = step * BT_STEP
        name = f"tcvae_week_{k:03d}.csv"
        path = os.path.join(current_csv_dir, name)

    if not os.path.exists(path):
        print(f"{name:<28} {'MISSING':>6}")
        all_ok = False
        continue
        if not os.path.exists(path):
            print(f"{name:<28} {'MISSING':>6}")
            all_ok = False
            continue

    df      = pd.read_csv(path, header=None)
    rows    = len(df)
    cols    = df.shape[1]
    p_j_sum = (df.iloc[:, 1].to_numpy() * df.iloc[:, 2].to_numpy()).sum()
        df      = pd.read_csv(path, header=None)
        rows    = len(df)
        cols    = df.shape[1]
        p_j_sum = (df.iloc[:, 1].to_numpy() * df.iloc[:, 2].to_numpy()).sum()

    # Compute max_mu exactly as optimizer/main.py get_mu_range() does:
    # expected eval price / optimizer initial_prices - 1
    probs    = df.iloc[:, 1].to_numpy() * df.iloc[:, 2].to_numpy()
    eval_p   = df.iloc[:, 3+Q : 3+2*Q].to_numpy()
    tree_ret = (eval_p * probs[:, None]).sum(axis=0) / initial_prices - 1
    top5     = sorted(tree_ret, reverse=True)[:5]
    max_mu   = (top5[0] * (1 - 4*0.01) + sum(top5[1:]) * 0.01) * 0.90
        # Compute max_mu exactly as optimizer/main.py get_mu_range() does:
        # expected eval price / optimizer initial_prices - 1
        probs    = df.iloc[:, 1].to_numpy() * df.iloc[:, 2].to_numpy()
        eval_p   = df.iloc[:, 3+Q : 3+2*Q].to_numpy()
        tree_ret = (eval_p * probs[:, None]).sum(axis=0) / initial_prices - 1
        top5     = sorted(tree_ret, reverse=True)[:5]
        max_mu   = (top5[0] * (1 - 4*0.01) + sum(top5[1:]) * 0.01) * 0.90

    ok = (rows == expected_rows and cols == expected_cols)
    status = "OK" if ok else "SHAPE MISMATCH"
    if not ok:
        all_ok = False
    print(f"{name:<28} {rows:>6} {cols:>5} {p_j_sum:>9.4f} {max_mu*100:>7.2f}%  {status}")
        ok = (rows == expected_rows and cols == expected_cols)
        status = "OK" if ok else "SHAPE MISMATCH"
        if not ok:
            all_ok = False
        print(f"{name:<28} {rows:>6} {cols:>5} {p_j_sum:>9.4f} {max_mu*100:>7.2f}%  {status}")

print()
print("All files OK" if all_ok else "WARNING: some files are missing or malformed")